# Private Knowledge Worker

A simple RAG-based assistant that answers questions from **your own** documents. Data stays local: embeddings via HuggingFace (no API key), vector store in Chroma, and you can use Ollama for the LLM to run fully offline.

**Concepts used:**
- **Week 5 (RAG):** Load docs → chunk → embed → store in Chroma → retrieve on query → answer with context.
- **Week 2 (tooling):** The LLM has a `search_knowledge(query)` tool and decides when to search; we handle `tool_calls` and feed results back (see step 6).

**Setup:** Point `KNOWLEDGE_BASE_PATH` to a folder of `.md` (and optionally `.txt`) files. By default we use the week5 knowledge base (../knowledge-base) so you can try it immediately.

## 1. Configuration and imports

In [ ]:
import os
import json
import glob
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
# Load documents path
_cwd = Path.cwd()
if (_cwd / "knowledge-base").exists():
    KNOWLEDGE_BASE_PATH = _cwd / "knowledge-base"
elif (_cwd / "week5" / "knowledge-base").exists():
    KNOWLEDGE_BASE_PATH = _cwd / "week5" / "knowledge-base"
elif (_cwd / ".." / ".." / "knowledge-base").exists():
    KNOWLEDGE_BASE_PATH = _cwd / ".." / ".." / "knowledge-base"
elif (_cwd / ".." / "knowledge-base").exists():
    KNOWLEDGE_BASE_PATH = _cwd / ".." / "knowledge-base"
else:
    KNOWLEDGE_BASE_PATH = _cwd / ".." / "knowledge-base"
KNOWLEDGE_BASE_PATH = KNOWLEDGE_BASE_PATH.resolve()
DB_NAME = "private_kb_chroma"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 5

## 2. Build the vector store (run once, or when you add new docs)
"**Setup:** Point \KNOWLEDGE_BASE_PATH\ to a folder of \.md\ (and optionally \.txt\) files. By default we use the week5 knowledge base (../knowledge-base) so you can try it immediately."

"**Pre-built vector store:** To skip building the index, [download private_kb_chroma.zip from Google Drive](https://drive.google.com/file/d/1vi87k5DtBIh8KliRxWPw9J9PtAKk-9mS/view?usp=sharing) and extract it into this folder."

In [ ]:
def load_and_chunk_documents(knowledge_path: Path):
    """Load all .md files from knowledge_path and split into chunks."""
    loader = DirectoryLoader(
        str(knowledge_path),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    docs = loader.load()
    # Add doc_type from parent folder name
    for doc in docs:
        doc.metadata["doc_type"] = Path(doc.metadata.get("source", "")).parent.name
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    return splitter.split_documents(docs)

chunks = load_and_chunk_documents(KNOWLEDGE_BASE_PATH)
print(f"Loaded {len(chunks)} chunks from {KNOWLEDGE_BASE_PATH}")

In [ ]:
# Embed with a local model (no API key). Uses HuggingFace all-MiniLM-L6-v2.
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
)
print(f"Vector store ready: {vectorstore._collection.count()} chunks")

## 3. LLM and retriever setup

In [ ]:
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-or-v1; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
MODEL= "gpt-4o-mini"

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

## 4. RAG chat with tooling (search_knowledge tool)

The LLM decides when to search: we pass a `search_knowledge(query)` tool, and when the model returns `tool_calls`, we run the retriever and feed the results back (Week 2 style).

In [ ]:
# System prompt: context is injected below; model can also call search_knowledge for extra lookups.
SYSTEM_PREFIX = """
You are a helpful assistant that answers questions using the provided context from a private knowledge base.
Relevant context from the knowledge base is provided below. Use it to answer the user's question.
You may also call search_knowledge(query) if you need to look up something else.
Answer only from the context provided or from search results. If the context does not contain the answer, say so. Do not make things up.
"""

# Tool schema for search_knowledge (Week 2 style)
search_knowledge_tool = {
    "type": "function",
    "function": {
        "name": "search_knowledge",
        "description": "Search the private knowledge base for relevant documents. Use this when you need to look up information about the company, employees, products, contracts, or other topics in the knowledge base.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query (e.g. a question or keywords to look up).",
                },
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    },
}

In [ ]:
tools = [search_knowledge_tool]

In [ ]:
def _to_text(value):
    """Extract a string from message or content (handles Gradio dict/object formats)."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        return value.get("text", value.get("content", str(value)))
    return getattr(value, "content", getattr(value, "text", str(value)))

def get_context(question: str) -> str:
    """Retrieve relevant chunks from the vector store and return as a single context string."""
    docs = retriever.invoke(question)
    if not docs:
        return "No relevant documents found."
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

def handle_tool_calls(message):
    """Execute tool calls and return a list of tool result messages for the API."""
    results = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "search_knowledge":
            try:
                args = json.loads(tool_call.function.arguments or "{}")
                query = args.get("query", "")
                content = get_context(query) if query else "No query provided."
            except Exception as e:
                content = f"Search error: {e}"
            results.append({
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            })
    return results

def chat(message, history):
    # Normalize inputs for Gradio (type="messages" can pass dicts or objects)
    text = _to_text(message)
    if not (text and text.strip()):
        return "Please enter a question."
    history = history or []
    history_roles = []
    for h in history:
        role = h.get("role", getattr(h, "role", "user"))
        content = _to_text(h.get("content", getattr(h, "content", "")))
        if isinstance(content, str):
            history_roles.append({"role": role, "content": content})
    # Always retrieve context for the user's question so the model has KB content (RAG works even if it doesn't call the tool)
    try:
        context = get_context(text)
    except Exception as e:
        return f"Retrieval error: {e}. Ensure the vector store cells were run and DB exists."
    system_with_context = SYSTEM_PREFIX + "\n\nRelevant context from the knowledge base:\n\n" + context
    messages = [{"role": "system", "content": system_with_context}] + history_roles + [{"role": "user", "content": text}]
    try:
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        while response.choices[0].finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            tool_results = handle_tool_calls(assistant_message)
            messages.append(assistant_message)
            messages.extend(tool_results)
            response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        return response.choices[0].message.content or ""
    except Exception as e:
        return f"Error: {e}. Check your API key, model name, and network."

## 5. Launch the Gradio chat UI

In [ ]:
gr.ChatInterface(fn=chat, type="messages", title="Private Knowledge Worker").launch(inbrowser=True)

---
## 6. How the tool loop works

1. **Tool schema:** `search_knowledge` is declared with a `query` parameter so the model knows when and how to call it.
2. **First call:** We send the user message with `tools=tools`. The model may return `finish_reason == "tool_calls"` and one or more tool calls.
3. **Handle tool calls:** For each tool call, we parse the arguments, run `get_context(query)`, and append `{"role": "tool", "content": ..., "tool_call_id": ...}` to the conversation.
4. **Loop:** We call the API again with the assistant message and tool results. The model then answers from the retrieved context. We repeat until `finish_reason` is not `tool_calls`.
See **week2/day4** for the same pattern with multiple tools.